In [41]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import ast

In [19]:
# Read in woke
wokeGames = pd.read_csv("Data/wokeGames.csv")
wokeGames.drop(columns=["Unnamed: 0"], inplace=True)
print(wokeGames.head())

         0
0   780290
1  1040390
2  2322440
3   296970
4  2027530
<class 'pandas.DataFrame'>


In [96]:
# Read in file
steam = pd.read_csv("Data/games_march2025_cleaned.csv")
steam = steam.drop(columns=["about_the_game", "release_date", "short_description",
                            "detailed_description", "support_url", "support_email",
                            "website", "reviews", "metacritic_score", "metacritic_url",
                            "notes", "score_rank", "header_image", "packages", "developers", "publishers",
                            "screenshots", "movies", "estimated_owners"])
steam["isWoke"] = steam["appid"].isin(wokeGames["0"])


In [100]:
def dummyVariable(name_of_feature):
    global steam
    steam["temp"] = steam[name_of_feature].apply(ast.literal_eval)
    exploded_languages = steam["temp"].explode()
    # turn many values of the same langauge from the explosion into "max", whether it supports that language or not
    dummies_languages = pd.get_dummies(exploded_languages).groupby(level=0).max()
    dummies_languages.columns += "_" + name_of_feature
    #print(dummies_languages.head())
    steam = pd.concat([steam, dummies_languages], axis=1)
    steam.drop(columns=["temp", name_of_feature], inplace=True)

dummyVariable("supported_languages")
dummyVariable("full_audio_languages")
dummyVariable("categories")
dummyVariable("genres")


In [110]:
print(steam.shape)
print(steam.head())
print(steam.info())
print(steam.iloc[50000].info())
steam.to_csv("test.csv")

(89618, 311)
    appid                             name  required_age  price  dlc_count  \
0     730                 Counter-Strike 2             0   0.00          1   
1  578080              PUBG: BATTLEGROUNDS             0   0.00          0   
2     570                           Dota 2             0   0.00          2   
3  271590        Grand Theft Auto V Legacy            17   0.00          0   
4  359550  Tom Clancy's Rainbow Six® Siege            17   3.99          9   

   windows    mac  linux  achievements  recommendations  ...  Short_genres  \
0     True  False   True             1          4401572  ...         False   
1     True  False  False            37          1732007  ...         False   
2     True   True   True             0            14337  ...         False   
3     True  False  False            77          1803063  ...         False   
4     True  False  False             0          1165929  ...         False   

   Simulation_genres  Software Training_genres  S

In [28]:
correlation_matrix = steam.corr()
plt.figure(figsize=(20,20))
sns.heatmap(correlation_matrix, annot=True, fmt="g")
plt.show()

ValueError: could not convert string to float: 'Counter-Strike 2'